# Random pruning · Dense-Aug / LFW
每个 seed 从同一份 checkpoint 出发，对 19 个卷积层各随机剪掉 50% 权重。一次性剪枝，无微调，0 张打分图，0 次反向传播。依次运行各格。

In [1]:
from pathlib import Path
import hashlib
import os
import sys
root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'src/occprune').is_dir())
os.chdir(root)
sys.path.insert(0, str(root / 'src'))
import numpy as np
import pandas as pd
import torch
from torch import nn
from occprune.data import load_pairs
from occprune.faceverif import evaluate_pairs
from occprune.pipeline import load_checkpoint

checkpoint = Path('checkpoints/dense_aug_seed42.pt')
assert hashlib.sha256(checkpoint.read_bytes()).hexdigest() == 'd8248b5dda150101f1b80188ad58d5b9b23977abd1843e58657f0cd5ef001bc3'
lfw = load_pairs('data/cache', 'lfw')
pairs = np.arange(len(lfw['images'])).reshape(-1, 2)
assert len(pairs) == len(lfw['issame']) == 6000
conditions = {'clean': None}
with np.load('results/masks/lfw_pair_occlusions.npz') as boxes:
    conditions.update({name: boxes[name] for name in ('square10', 'square30', 'square50')})

### 逐层生成随机掩码
每轮只初始化一次 CPU 随机数生成器。直接抽取该层一半的位置，将掩码应用于卷积核，并核对被选中的位置和总数。

In [2]:
@torch.no_grad()
def prune_random(model, seed):
    generator = torch.Generator().manual_seed(seed)
    layers = [(name, layer) for name, layer in model.named_modules()
              if isinstance(layer, nn.Conv2d) and layer is not model.conv1]
    assert len(layers) == 19
    audit = []
    for name, layer in layers:
        weight = layer.weight
        size = weight.numel()
        selected = torch.randperm(size, generator=generator)[:size // 2]
        keep = torch.ones(size, dtype=torch.bool)
        keep[selected] = False
        assert 2 * int((~keep).sum()) == size
        keep = keep.reshape(weight.shape).to(weight.device)
        weight.masked_fill_(~keep, 0)
        assert torch.count_nonzero(weight[~keep]).item() == 0
        audit.append(dict(seed=seed, layer=name, weights=size, masked=size // 2))
    assert sum(row['weights'] for row in audit) == 11157504
    return audit

### 先测未剪枝基线，再测三个种子
每次重新加载原始模型，沿用同一批照片对、遮挡框、水平翻转与十折评测。输出准确率的单位为 %。

In [3]:
records, audit = [], []
for seed in (None, 42, 43, 44):
    model, head, meta = load_checkpoint(checkpoint)
    label = 'dense' if seed is None else f'random_{seed}'
    if seed is not None:
        audit.extend(prune_random(model, seed))
    print(label, next(model.parameters()).device, flush=True)
    scores = evaluate_pairs(model, lfw['images'], pairs, lfw['issame'],
                            conditions=conditions, batch_size=64, amp='none')
    records.append({'run': label, **{name: 100 * scores[name]['acc'] for name in conditions}})
    print(records[-1], flush=True)
    del model, head
accuracy = pd.DataFrame(records).set_index('run')
print(accuracy.round(3).to_string())

dense mps:0
{'run': 'dense', 'clean': 98.76666666666667, 'square10': 98.76666666666667, 'square30': 96.48333333333333, 'square50': 90.88333333333333}
random_42 mps:0
{'run': 'random_42', 'clean': 64.64999999999999, 'square10': 60.90000000000001, 'square30': 54.33333333333332, 'square50': 51.983333333333334}
random_43 mps:0
{'run': 'random_43', 'clean': 64.88333333333333, 'square10': 62.85000000000001, 'square30': 55.65, 'square50': 52.58333333333333}
random_44 mps:0
{'run': 'random_44', 'clean': 65.10000000000001, 'square10': 63.13333333333334, 'square30': 56.48333333333333, 'square50': 51.26666666666666}
            clean  square10  square30  square50
run                                            
dense      98.767    98.767    96.483    90.883
random_42  64.650    60.900    54.333    51.983
random_43  64.883    62.850    55.650    52.583
random_44  65.100    63.133    56.483    51.267


### 汇总
标准差使用三个剪枝 seed 的样本标准差；下降量为同机 dense 准确率减去随机剪枝均值（百分点）。结果另存到 results/random50。

In [4]:
random_runs = accuracy.loc[['random_42', 'random_43', 'random_44']]
summary = pd.DataFrame({'dense_pct': accuracy.loc['dense'], 'random_mean_pct': random_runs.mean(),
                        'seed_sd_pp': random_runs.std(ddof=1),
                        'drop_pp': accuracy.loc['dense'] - random_runs.mean()})
output = Path('results/random50')
output.mkdir(parents=True, exist_ok=True)
accuracy.to_csv(output / 'lfw_accuracy.csv')
summary.to_csv(output / 'lfw_summary.csv')
pd.DataFrame(audit).to_csv(output / 'layer_audit.csv', index=False)
print(summary.round(3).to_string())

          dense_pct  random_mean_pct  seed_sd_pp  drop_pp
clean        98.767           64.878       0.225   33.889
square10     98.767           62.294       1.216   36.472
square30     96.483           55.489       1.084   40.994
square50     90.883           51.944       0.659   38.939
